한 HWP 세션에는 여러 문서가 동시에 열려 있을 수 있습니다. hwpapi 는 이를 **`app.documents` 컬렉션** 과 **`app.use_document(...)` 컨텍스트 매니저** 로 다룹니다.

## `app.documents` — 문서 컬렉션

파이썬 리스트처럼 동작합니다.

```python
from hwpapi.core import App
app = App(is_visible=True)

# 열린 문서 개수
print(len(app.documents))

# 인덱스 접근
first = app.documents[0]
last  = app.documents[-1]

# 순회
for doc in app.documents:
    print(doc.full_name, doc.modified)

# 활성 문서
active = app.documents.active
```

## `Document` — 개별 문서 핸들

`app.documents[i]` 가 반환하는 객체는 `Document` 인스턴스입니다.

| 속성 | 설명 |
|:---|:---|
| `full_name` | 전체 경로 포함 파일명 (저장 안 된 문서는 `""`) |
| `path` | 문서 폴더 경로 |
| `modified` | 마지막 저장 이후 수정 여부 (bool) |
| `document_id` | HWP 내부 ID |
| `edit_mode` | 편집 모드 (0=일반, 1=읽기 전용 등) |
| `format` | 문서 포맷 문자열 |
| `raw` | 원시 `IXHwpDocument` COM 객체 |

| 메소드 | 설명 |
|:---|:---|
| `activate()` | 이 문서를 활성창으로 전환 |
| `save()` | 현재 경로에 저장 |
| `save_as(path, format=None)` | 다른 이름/포맷으로 저장 |
| `close(save=False)` | 이 문서만 닫기 |
| `clear()` | 문서 내용 비우기 |
| `undo()` / `redo()` | 문서별 undo/redo |

## xlwings 스타일 — `doc.xxx()` 로 바로 쓰기

`Document` 는 **App의 모든 메소드를 자동으로 위임** 받습니다. `with app.use_document(...)` 없이 바로 `doc.insert_text(...)` 처럼 쓸 수 있습니다. 내부적으로는 호출 시점에 자동으로 해당 문서를 활성창으로 전환한 뒤 App 메소드를 실행하고, 원래 활성 문서로 복원합니다.

xlwings 의 ``book.sheets[0].range(...)`` 스타일을 그대로 따른 설계입니다.

```python
# 전통적 방식 — 명시적 switch
with app.use_document(doc):
    app.insert_text("...")
    app.find_text("키워드")

# xlwings 스타일 — doc 에서 바로 호출
doc.insert_text("...")              # 자동 switch
doc.find_text("키워드")             # 자동 switch
doc.set_charshape(bold=True)        # 자동 switch
doc.replace_all("a", "b")           # 자동 switch
doc.styled_text("강조", bold=True)  # 자동 switch
doc.save()                          # Document 자체 메소드
```

### 접근자도 위임됩니다

`doc.move`, `doc.actions`, `doc.api` 등 App 의 속성도 그대로 쓸 수 있습니다. 접근 시 해당 문서가 활성화된 뒤 접근자가 반환됩니다.

```python
doc.move.top_of_file()
doc.move.bottom_of_file()
doc.actions.SelectAll.run()
doc.actions.TableCreate.pset.Rows = 3
doc.api.Run("BreakPara")
```

### 여러 문서 순회 → 동일 작업 반복

`use_document` 없이도 깔끔합니다.

```python
paths = ["r1.hwp", "r2.hwp", "r3.hwp"]
docs = [app.documents.open(p) for p in paths]

for doc in docs:
    n = doc.replace_all("2025년", "2026년")
    print(f"{doc.full_name}: {n}건 치환")
    doc.save()
```

### 독립적 문서간 데이터 이동

```python
source = app.documents.open("data.hwp")
target = app.documents.add()

# source 에서 읽기
from hwpapi import constants as const
raw_text = source.get_text(
    spos=const.ScanStartPosition.Document,
    epos=const.ScanEndPosition.Document,
)

# target 에 요약 작성
target.insert_text("자동 요약\n\n")
target.styled_text("원본 길이: ", bold=True)
target.insert_text(f"{len(raw_text)}자\n")
target.save_as("summary.hwp")
```

::: {.callout-note}
**구현 원리**: `Document.__getattr__` 가 App 의 메소드/속성을 찾아 자동으로 활성화 래퍼를 감쌉니다.

- **메소드 (함수/builtin)** — 호출 시 `with app.use_document(self):` 로 감싸서 실행 후 원래 활성 문서 복원
- **접근자 인스턴스** (MoveAccessor 등) — 해당 문서 활성화 후 접근자 반환
:::

## 새 문서 추가 / 파일 열기

```python
# 빈 새 문서 (같은 창의 새 탭)
new_doc = app.documents.add()

# 별도 창에 새 문서
new_doc = app.documents.add(is_tab=False)

# 기존 파일 열기 (현재 세션에 추가)
opened = app.documents.open("C:/report.hwp")
```

## `use_document` — 임시 활성 전환

다중 문서 작업의 **핵심 패턴**. 블록 내부에서만 지정된 문서가 활성창이 되고, 블록이 끝나면 원래 활성 문서로 돌아옵니다.

```python
# 인덱스로 지정
with app.use_document(0):
    app.insert_text("첫 번째 문서에 추가")

# Document 객체로 지정
other = app.documents.open("other.hwp")
with app.use_document(other):
    app.replace_all("OLD", "NEW")
    other.save()
```

## 실전 패턴 1 — 모든 문서에 같은 치환 적용

```python
app = App()

# 3개 파일을 순서대로 연다
paths = ["report_1.hwp", "report_2.hwp", "report_3.hwp"]
docs = [app.documents.open(p) for p in paths]

# 각 문서에 같은 치환 적용
for doc in docs:
    with app.use_document(doc):
        n = app.replace_all("2025년", "2026년")
        print(f"{doc.full_name}: {n}건 치환")
        doc.save()
```

## 실전 패턴 2 — 문서 A 에서 내용 읽어서 문서 B 에 표로 삽입

```python
from hwpapi import constants as const

app = App()
source = app.documents.open("data.hwp")
summary = app.documents.add()   # 빈 새 문서

# 1. 원본에서 전체 텍스트 읽기
with app.use_document(source):
    raw = app.get_text(
        spos=const.ScanStartPosition.Document,
        epos=const.ScanEndPosition.Document,
    )

# 2. 파이썬에서 처리 — 예: 숫자만 추출
import re
numbers = re.findall(r"\d[\d,]+", raw)

# 3. 요약 문서에 표로 삽입
with app.use_document(summary):
    app.insert_text("추출된 숫자 요약\n\n")
    a = app.actions.TableCreate
    a.pset.Rows = len(numbers) + 1
    a.pset.Cols = 2
    a.run()
    app.insert_text("번호");  app.api.Run("TableRightCell")
    app.insert_text("값");    app.api.Run("TableRightCell")
    for i, n in enumerate(numbers, 1):
        app.insert_text(str(i)); app.api.Run("TableRightCell")
        app.insert_text(n);       app.api.Run("TableRightCell")
    summary.save_as("summary.hwp")
```

## 실전 패턴 3 — 전체 일괄 저장/닫기

```python
# 열린 모든 문서 저장
saved = app.documents.save_all()
print(f"{saved}개 문서 저장됨")

# 전체 닫기 (저장 없이)
closed = app.documents.close_all(save=False)

# 전체 저장 후 닫기
app.documents.save_all()
app.documents.close_all(save=True)
```

## 실전 패턴 4 — 이름으로 특정 문서 찾기

```python
# 파일 경로에 '2026' 이 포함된 첫 문서 찾기
doc = app.documents.find("2026")
if doc:
    with app.use_document(doc):
        app.insert_text("추가 내용")
        doc.save()
```

## 정리 — 언제 어떤 API?

| 상황 | 추천 패턴 |
|:---|:---|
| 하나의 문서만 다룬다 | `app.insert_text()`, `app.save()` — 그냥 App 메소드 |
| 문서 정보 조회 | `doc.full_name`, `doc.modified` |
| 문서 간 전환 | `with app.use_document(doc):` |
| 여러 문서에 동일 작업 | `for doc in app.documents: ...` |
| 전체 일괄 저장/닫기 | `app.documents.save_all() / close_all()` |
| 이름으로 찾기 | `app.documents.find("...")` |
| 다른 창 분리 | `app.documents.add(is_tab=False)` |